# PR Final — Holistic Data Preparer

## Part B — Data Acquisition

In [1]:
import os
import json
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests
rng = np.random.default_rng(42)

from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import (
    LabelEncoder, OrdinalEncoder, OneHotEncoder,
    StandardScaler, MinMaxScaler, MaxAbsScaler,
    RobustScaler, Normalizer, KBinsDiscretizer,
    FunctionTransformer, PowerTransformer, Binarizer
)
from sklearn.compose import ColumnTransformer
from sklearn.cluster import KMeans

BASE = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA = os.path.join(BASE, "Dataset")
OUT = os.path.join(BASE, "Final Dataset")
os.makedirs(OUT, exist_ok=True)

df = pd.read_csv(os.path.join(DATA, "customer_credit_risk.csv"))
manager = pd.read_json(os.path.join(DATA, "customer_manager.json"))

conn = sqlite3.connect(os.path.join(DATA, "loan_repayment.db"))
repayment = pd.read_sql_query("SELECT * FROM loan_repayment", conn)
conn.close()

print(df.shape)
print(manager.shape)
print(repayment.shape)

(220, 15)
(80, 4)
(150, 3)


In [2]:
try:
    api_url = "https://api.worldbank.org/v2/country/IND/indicator/NY.GDP.MKTP.CD?format=json&per_page=5"
    response = requests.get(api_url, timeout=10)
    response.raise_for_status()
    api_data = response.json()
    print("API request: successful")
except Exception as e:
    api_data = []
    print("API request skipped:", type(e).__name__)

API request: successful


## Part C — Data Understanding & Cleaning

In [3]:
print(df.head())
print(df.info())
print(df.describe(include="all").T)

  customer_id   age  gender   region education_level employment_type  \
0      C10001  42.0  Female    North   Post-Graduate        Salaried   
1      C10002  43.0  Female  Central   Post-Graduate        Business   
2      C10003  23.0    Male    North     High School        Salaried   
3      C10004  41.0   Other     East        Graduate        Business   
4      C10005  41.0   Other    South        Graduate   Self-Employed   

   annual_income  loan_amount loan_purpose  credit_score  repayment_history  \
0      935320.78    277028.12        Other         579.0                 26   
1      558804.45    204240.96     Business         712.0                 17   
2     1060126.33    656982.47        Other         673.0                 11   
3     1113358.12    176696.86    Education         829.0                 24   
4      303710.15    108621.11          Car         688.0                 24   

   transaction_count  spending_ratio   join_date  default_flag  
0                 44       

In [4]:
print("Missing values:")
print(df.isna().sum())

print("\nDuplicates:", df.duplicated().sum())

Missing values:
customer_id          0
age                  3
gender               2
region               2
education_level      0
employment_type      2
annual_income        4
loan_amount          0
loan_purpose         0
credit_score         3
repayment_history    0
transaction_count    0
spending_ratio       0
join_date            0
default_flag         0
dtype: int64

Duplicates: 0


In [6]:
# Profiling-style quality report using pandas
profile = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "missing_%": (df.isna().mean()*100).round(2),
    "unique": df.nunique(dropna=True)
})
profile

,dtype,missing,missing_%,unique
customer_id,object,0,0.00,220
age,float64,3,1.36,44
gender,object,2,0.91,3
region,object,2,0.91,5
education_level,object,0,0.00,4
employment_type,object,2,0.91,4
annual_income,float64,4,1.82,216
loan_amount,float64,0,0.00,219
loan_purpose,object,0,0.00,5
credit_score,float64,3,1.36,150


In [7]:
# Simple imputer
num_cols = ["age", "annual_income", "credit_score"]
cat_cols = ["gender", "region", "employment_type"]

num_imputer = SimpleImputer(strategy="mean")
cat_imputer = SimpleImputer(strategy="most_frequent")

df_clean = df.copy()
df_clean[num_cols] = num_imputer.fit_transform(df_clean[num_cols])
df_clean[cat_cols] = cat_imputer.fit_transform(df_clean[cat_cols])

print(df_clean[num_cols + cat_cols].isna().sum())

age                0
annual_income      0
credit_score       0
gender             0
region             0
employment_type    0
dtype: int64


In [8]:
# Missing indicator + random sample imputation
sample_df = df.copy()
sample_df["income_missing"] = sample_df["annual_income"].isna().astype(int)

for col in ["annual_income", "credit_score"]:
    missing = sample_df[col].isna()
    values = sample_df.loc[~missing, col]
    if missing.any():
        sample_df.loc[missing, col] = rng.choice(values.to_numpy(), missing.sum(), replace=True)

print(sample_df[["annual_income", "credit_score", "income_missing"]].isna().sum())

annual_income     0
credit_score      0
income_missing    0
dtype: int64


In [9]:
# KNN imputation
knn_cols = ["age", "annual_income", "loan_amount", "credit_score", "transaction_count", "spending_ratio"]
knn = KNNImputer(n_neighbors=5)
knn_result = pd.DataFrame(knn.fit_transform(df[knn_cols]), columns=knn_cols)
knn_result.head()

,age,annual_income,loan_amount,credit_score,transaction_count,spending_ratio
0,42.0,935320.78,277028.12,579.0,44.0,0.319
1,43.0,558804.45,204240.96,712.0,30.0,0.304
2,23.0,1060126.33,656982.47,673.0,37.0,0.208
3,41.0,1113358.12,176696.86,829.0,39.0,0.855
4,41.0,303710.15,108621.11,688.0,36.0,0.422


In [10]:
# MICE / Iterative Imputation
mice = IterativeImputer(random_state=42, max_iter=10)
mice_result = pd.DataFrame(mice.fit_transform(df[knn_cols]), columns=knn_cols)
mice_result.head()

,age,annual_income,loan_amount,credit_score,transaction_count,spending_ratio
0,42.0,935320.78,277028.12,579.0,44.0,0.319
1,43.0,558804.45,204240.96,712.0,30.0,0.304
2,23.0,1060126.33,656982.47,673.0,37.0,0.208
3,41.0,1113358.12,176696.86,829.0,39.0,0.855
4,41.0,303710.15,108621.11,688.0,36.0,0.422


In [11]:
# Complete Case Analysis
cca = df.dropna()
print("Original:", df.shape)
print("After CCA:", cca.shape)

Original: (220, 15)
After CCA: (204, 15)


## Part D — Outlier Handling

In [12]:
outlier_cols = ["annual_income", "loan_amount", "credit_score"]

# Z-score
z = (df_clean[outlier_cols] - df_clean[outlier_cols].mean()) / df_clean[outlier_cols].std()
print("Z-score outliers:")
print((z.abs() > 3).sum())

# IQR
for col in outlier_cols:
    q1 = df_clean[col].quantile(.25)
    q3 = df_clean[col].quantile(.75)
    iqr = q3 - q1
    mask = (df_clean[col] < q1 - 1.5*iqr) | (df_clean[col] > q3 + 1.5*iqr)
    print(col, "IQR:", mask.sum())

# Percentile
for col in outlier_cols:
    low, high = df_clean[col].quantile([.01, .99])
    print(col, "percentile limits:", round(low,2), round(high,2))

# Winsorization
winsor = df_clean.copy()
for col in outlier_cols:
    low, high = winsor[col].quantile([.01, .99])
    winsor[col] = winsor[col].clip(low, high)

winsor[outlier_cols].describe().T

Z-score outliers:
annual_income    1
loan_amount      1
credit_score     1
dtype: int64
annual_income IQR: 2
loan_amount IQR: 2
credit_score IQR: 1
annual_income percentile limits: 315611.77 1448960.27
loan_amount percentile limits: 75095.38 981905.12
credit_score percentile limits: 523.42 841.1


,count,mean,std,min,25%,50%,75%,max
annual_income,220.0,842645.168383,245855.989254,315611.7747,664409.875,841941.085,1.001025e+06,1.448960e+06
loan_amount,220.0,412987.372809,213596.460377,75095.3832,250134.700,379551.695,5.500029e+05,9.819051e+05
credit_score,220.0,695.861343,70.573349,523.4200,644.250,700.000,7.450000e+02,8.411000e+02


## Part E — Feature Engineering

In [13]:
# Mixed variables and date/time
df_fe = winsor.copy()
df_fe["join_date"] = pd.to_datetime(df_fe["join_date"])
df_fe["join_year"] = df_fe["join_date"].dt.year
df_fe["join_month"] = df_fe["join_date"].dt.month
df_fe["join_day"] = df_fe["join_date"].dt.day
df_fe["join_weekday"] = df_fe["join_date"].dt.day_name()

# Ordinal encoding
education_order = [["High School", "Graduate", "Post-Graduate", "PhD"]]
ord_enc = OrdinalEncoder(categories=education_order)
df_fe["education_encoded"] = ord_enc.fit_transform(df_fe[["education_level"]])

# Label encoding
label_enc = LabelEncoder()
df_fe["gender_encoded"] = label_enc.fit_transform(df_fe["gender"].astype(str))

# One-hot encoding
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
ohe_array = ohe.fit_transform(df_fe[["region", "loan_purpose"]])
ohe_cols = ohe.get_feature_names_out(["region", "loan_purpose"])
ohe_df = pd.DataFrame(ohe_array, columns=ohe_cols, index=df_fe.index)
df_fe = pd.concat([df_fe, ohe_df], axis=1)

print(df_fe.shape)

(220, 31)


In [14]:
# Numerical feature encoding
df_fe["income_bin"] = pd.cut(
    df_fe["annual_income"],
    bins=[0, 500000, 1000000, 1500000, np.inf],
    labels=["Low", "Medium", "High", "Very High"]
)

df_fe["high_loan_flag"] = Binarizer(threshold=750000).fit_transform(
    df_fe[["loan_amount"]]
).ravel()

quantile_bin = KBinsDiscretizer(n_bins=4, encode="ordinal", strategy="quantile")
df_fe["credit_quantile_bin"] = quantile_bin.fit_transform(df_fe[["credit_score"]]).ravel()

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df_fe["income_kmeans_bin"] = kmeans.fit_predict(df_fe[["annual_income"]])

df_fe[["income_bin", "high_loan_flag", "credit_quantile_bin", "income_kmeans_bin"]].head()

,income_bin,high_loan_flag,credit_quantile_bin,income_kmeans_bin
0,Medium,0.0,0.0,0
1,Medium,0.0,2.0,1
2,High,0.0,1.0,0
3,High,0.0,3.0,0
4,Low,0.0,1.0,1


## Part F — Feature Scaling

In [14]:
scale_cols = ["annual_income", "loan_amount", "credit_score"]

standard = StandardScaler().fit_transform(df_fe[scale_cols])
normalized = Normalizer().fit_transform(df_fe[scale_cols])
minmax = MinMaxScaler().fit_transform(df_fe[scale_cols])
maxabs = MaxAbsScaler().fit_transform(df_fe[scale_cols])
robust = RobustScaler().fit_transform(df_fe[scale_cols])

print("Standard:", standard[:2])
print("Normalized:", normalized[:2])
print("MinMax:", minmax[:2])
print("MaxAbs:", maxabs[:2])
print("Robust:", robust[:2])

Standard: [[ 0.37781044 -0.6379755  -1.65966115]
 [-1.15713276 -0.97952213  0.2292007 ]]
Normalized: [[9.58826968e-01 2.83990304e-01 5.93551246e-04]
 [9.39230432e-01 3.43285250e-01 1.19671930e-03]]
MinMax: [[0.54679475 0.22268479 0.17495593]
 [0.2145789  0.1424175  0.59361622]]
MaxAbs: [[0.64551168 0.28213329 0.68838426]
 [0.38565892 0.20800478 0.84651052]]
Robust: [[ 0.27740787 -0.34189543 -1.20099256]
 [-0.84112858 -0.58462592  0.1191067 ]]


## Part G — Feature Construction & Transformation

In [15]:
# Feature construction
df_fe["debt_to_income_ratio"] = df_fe["loan_amount"] / df_fe["annual_income"]
df_fe["average_monthly_transactions"] = df_fe["transaction_count"] / 12
df_fe["spending_to_income_ratio"] = df_fe["spending_ratio"] * df_fe["annual_income"]

# FunctionTransformer
positive_income = df_fe[["annual_income"]].clip(lower=1)

log_transform = FunctionTransformer(np.log1p)
sqrt_transform = FunctionTransformer(np.sqrt)
reciprocal_transform = FunctionTransformer(lambda x: 1 / x)

df_fe["income_log"] = log_transform.transform(positive_income).to_numpy().ravel()
df_fe["income_sqrt"] = sqrt_transform.transform(positive_income).to_numpy().ravel()
df_fe["income_reciprocal"] = reciprocal_transform.transform(positive_income).to_numpy().ravel()

print(df_fe[[
    "debt_to_income_ratio",
    "average_monthly_transactions",
    "spending_to_income_ratio",
    "income_log",
    "income_sqrt",
    "income_reciprocal"
]].head())

   debt_to_income_ratio  average_monthly_transactions  \
0              0.296185                      3.666667   
1              0.365496                      2.500000   
2              0.619721                      3.083333   
3              0.158706                      3.250000   
4              0.344161                      3.000000   

   spending_to_income_ratio  income_log  income_sqrt  income_reciprocal  
0             298367.328820   13.748646   967.119837       1.069152e-06  
1             169876.552800   13.233557   747.532240       1.789535e-06  
2             220506.276640   13.873900  1029.624364       9.432838e-07  
3             951921.192600   13.922892  1055.157865       8.981836e-07  
4             133188.168923   12.662271   561.793356       3.168450e-06  


In [16]:
# PowerTransformer
positive_cols = df_fe[["annual_income", "loan_amount"]].clip(lower=1)

box_cox = PowerTransformer(method="box-cox")
yeo_johnson = PowerTransformer(method="yeo-johnson")

boxcox_result = box_cox.fit_transform(positive_cols)
yj_result = yeo_johnson.fit_transform(df_fe[["annual_income", "loan_amount"]])

print("Box-Cox:", boxcox_result[:2])
print("Yeo-Johnson:", yj_result[:2])

Box-Cox: [[ 0.4116676  -0.5368665 ]
 [-1.1700261  -0.99819536]]
Yeo-Johnson: [[ 0.41166759 -0.53686665]
 [-1.1700261  -0.99819558]]


In [17]:
# ColumnTransformer
numeric_features = ["annual_income", "loan_amount", "credit_score"]
categorical_features = ["gender", "region", "loan_purpose"]

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features)
])

combined = preprocessor.fit_transform(df_fe)
print("ColumnTransformer output shape:", combined.shape)

ColumnTransformer output shape: (220, 16)


## Part H — Final Deliverable

In [18]:
final_cols = [
    "customer_id", "age", "gender", "region", "education_level",
    "employment_type", "annual_income", "loan_amount", "loan_purpose",
    "credit_score", "repayment_history", "transaction_count",
    "spending_ratio", "join_date", "default_flag",
    "join_year", "join_month", "join_day", "join_weekday",
    "education_encoded", "gender_encoded", "income_bin",
    "high_loan_flag", "credit_quantile_bin", "income_kmeans_bin",
    "debt_to_income_ratio", "average_monthly_transactions",
    "spending_to_income_ratio", "income_log", "income_sqrt",
    "income_reciprocal"
]

final_df = df_fe[final_cols].copy()
final_path = os.path.join(OUT, "final_customer_credit_risk.csv")
final_df.to_csv(final_path, index=False)

print("Final shape:", final_df.shape)
print("Missing values:", int(final_df.isna().sum().sum()))
print("Saved:", final_path)

Final shape: (220, 31)
Missing values: 0
Saved: d:\AI_ML_PYTHON\Data Preprocessing and Feature Engineering\PR Final Exam Holistic Data Preparer\Final Dataset\final_customer_credit_risk.csv
